# Tracey letlape | LTLTRA001 | CSC3042F | A2

#### Imports and Necessary Libraries

In [ ]:
import torch
import torch.nn
from torch.utils.data import Dataset, DataLoader, TensorDataset


import matplotlib as plt
import pandas as pd
import os

## 1. Data Processing

### a. Building character and phoneme vocabulary from training data
* Define a *read_raw_data* function that takes a file_name as input and returns a DataFrame of the words and phonemes.
* Define a *build_vocab* function that builds a vocab depending on the bool value of the *char* and *phoneme* function parameters.
    * Vocab is represented as a dictionary of characters/phonesmes as strings and integers as values

In [ ]:
DATA_DIR = "./a2-dataset"

def read_raw_data(file_name: str) -> pd.DataFrame:
    """
    Read the data from the file_name with the assumption that the file is a csv_file.
    Parameters:
        file_name: Name of a CSV file 
    Returns:
        df: DataFrame of the words and phonemes
    """
    file_path = os.path.join(DATA_DIR, file_name)
    df = pd.read_csv(file_path)
    return df

def build_vocab(data_df: pd.DataFrame, char: bool = True, phoneme: bool = False) -> dict:
    """
    Builds a vocabulary depending on the bool value of char and phoneme parameters.
    If both char and phoneme are true, returns a vocab with only the special tokens.
    Parameters:
        data_df: Represents the DataFrame of the words and phonemes
        char: Builds a char_vocab if True
        phoneme: Builds a phoneme_vocab if True
    Returns:
        vocab: A dictionary of a word/phoneme as key and index as value
    """
    vocab = {
        "<PAD>": 0,
        "<SOS>": 1,
        "<EOS>": 2,
        "<UNK>": 3
    }
    idx = 4
    if char:
        words = data_df['word'].tolist()
        for word in words:
            idx = _build_char_vocab_helper(word, vocab, idx)
    elif phoneme:
        phonemes = data_df['phonemes'].tolist()
        for ph in phonemes:
            ph = ph.split(" ")
            idx = _build_phoneme_vocab_helper(ph, vocab, idx)
    return vocab
    

def _build_char_vocab_helper(word: str, vocab: dict[str, int], idx: int) -> int:
    for c in word:
        if c not in vocab:
            vocab[c] = idx
            idx += 1
    return idx

def _build_phoneme_vocab_helper(phonemes: list[str], vocab: dict[str, int], idx: int) -> int:
    for phoneme in phonemes:
        if phoneme not in vocab:
            vocab[phoneme] = idx
            idx += 1
    return idx


### b. Implementation of a PyTorch Dataset class that for encoding (word, phoneme) pairs as integer index sequences


In [ ]:
class WPDataset(Dataset):
    def __init__(self, pairs: tuple[str, str], chr_vocab: dict[str, int], ph_vocab: dict[str, int]):
        self.pairs: tuple[str, str] = pairs
        self.chr_vocab: dict[str, int] = chr_vocab
        self.ph_vocab: dict[str, int] = ph_vocab

        self._cache: dict[int, tuple[list[int], list[int]]] = {}

    def __len__(self) -> int:
        """
        Returns the length of the dataset.
        """
        return len(self.pairs)
    
    def __getitem__(self, idx: int) -> tuple[list[int], list[int]]:
        """
        Returns the encoded pair at the given index
        Parameters:
            idx: Represents an index
        Returns:
            enc_word, enc_phoneme
        """
        if idx not in self._cache:
            word, phoneme = self.pairs[idx]
            self._cache[idx] = self.encode_pair(word, phoneme)
        return self._cache[idx]

    def encode_pair(self, word: str, phoneme: str) -> tuple[list[int], list[int]]:
        """
        Encodes the given word and phoneme pair
        Parameters:
            word: Represents a word in the dataset.
            phoneme: Represents a phoneme corresponding to the word.
        Returns:
            enc_word, enc_phoneme
        """
        word_enc = []
        phoneme_enc = [self.ph_vocab["<SOS>"]]

        self._encode_word(word, self.chr_vocab, word_enc)
        self._encode_phoneme(phoneme.split(" "), self.ph_vocab, phoneme_enc)

        return word_enc, phoneme_enc
    
    def decode_pair(self, enc_word: list[int], enc_ph: list[int]) -> tuple[str, list[str]]:
        """
        Decodes the given encoded word and phoneme pairs
        Parameters:
            enc_word: Represent a word encoded as integer sequences
            enc_ph: Represent a phoneme encoded as integer sequences
        Returns:
            dec_word, dec_ph
        """
        # Invert the vocabs
        inv_chr_vocab: dict[int, str] = {v: k for k, v in self.chr_vocab.items()}
        inv_ph_vocab: dict[int, str] = {v: k for k, v in self.ph_vocab.items()}

        dec_word = self._decode_word(enc_word, inv_chr_vocab)
        dec_ph = self._decode_phoneme(enc_ph, inv_ph_vocab)
        
        return dec_word, dec_ph

    def _encode_word(self, word: str, chr_vocab: dict, int_seq: list) -> None:
        for c in word:
            int_seq.append(self.chr_vocab.get(c, chr_vocab["<UNK>"]))

    def _encode_phoneme(self, phoneme: list, ph_vocab: dict[str, int], int_seq: list[int]) -> None:
        for ph in phoneme:
            int_seq.append(ph_vocab.get(ph, ph_vocab["<UNK>"]))
        int_seq.append(ph_vocab["<EOS>"])

    def _decode_word(self, enc_word: list, inv_chr_vocab: dict[int, str]) -> str:
        return "".join(inv_chr_vocab[enc] for enc in enc_word)
    
    def _decode_phoneme(self, enc_ph: list[int], inv_ph_vocab: dict[int, str]) -> list[str]:
        return [inv_ph_vocab[enc] for enc in enc_ph]